# MeatGuard II — 00 · Teacher training and selection

Trains the **knowledge-distillation teacher** for MeatGuard-S and selects it without touching the test set.

**Protocol (must match the paper):**
- Inputs: SAM-cropped MeatScan images (`crops/`), split by **capture session** (`sessions.csv`), so no near-duplicate
  of a test image is ever seen in training.
- Candidates: ImageNet-pretrained `efficientnet_b0`, `convnext_tiny`, `swin_t`, `resnet50` (torchvision), all
  fine-tuned with the same recipe.
- Selection: highest **validation macro-F1** (ties: spoiled recall, then fewer parameters).
- The **test split is evaluated once**, for the selected teacher only, with 95% CIs that resample *sessions*.

All training logic lives in `src/train_teacher.py`; this notebook only drives it, so a run here and a run from the
command line are identical. Runs are resumable — re-running a cell after a Colab disconnect continues from `last.pt`.

## 1. Environment

In [ ]:
# Colab: mount Drive so data and results survive disconnects (skip when running locally)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# --- paths: edit these -------------------------------------------------------------
REPO_URL  = "https://github.com/Berchie-Sam/MeatGuard_II.git"
REPO_BRANCH = "session-split-teacher"   # branch with the session split, teacher training and pruning
REPO_DIR  = "/content/MeatGuard_II" if IN_COLAB else os.path.abspath("..")
# Folder (or .zip) that contains  crops/<class>/*.jpg  and  sessions.csv
DATA_SRC  = "/content/drive/MyDrive/MeatGuard/full.zip" if IN_COLAB else "E:/Datasets/full"
DATA_ROOT = "/content/data/full" if IN_COLAB else DATA_SRC          # fast local copy on Colab
RUNS_DIR  = "/content/drive/MyDrive/MeatGuard/runs/teacher" if IN_COLAB else os.path.join(REPO_DIR, "runs/teacher")
ARCHS     = ["efficientnet_b0", "convnext_tiny", "swin_t", "resnet50"]
EPOCHS, BATCH, WORKERS = 30, 64, 4
# ---------------------------------------------------------------------------------------

In [ ]:
if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU visible - training will be very slow"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. Data
Copy the data to the Colab VM's local disk once (reading thousands of small files from Drive is slow). The dataset
must be **complete**: the trainer refuses to run if images listed in `sessions.csv` are missing.

In [ ]:
import zipfile, shutil
if IN_COLAB and not os.path.exists(os.path.join(DATA_ROOT, "sessions.csv")):
    os.makedirs(os.path.dirname(DATA_ROOT), exist_ok=True)
    if DATA_SRC.endswith(".zip"):
        with zipfile.ZipFile(DATA_SRC) as z:
            z.extractall(os.path.dirname(DATA_ROOT))
    else:
        shutil.copytree(DATA_SRC, DATA_ROOT)
CROPS, SPLIT = os.path.join(DATA_ROOT, "crops"), os.path.join(DATA_ROOT, "sessions.csv")
print(CROPS, os.path.exists(CROPS), "|", SPLIT, os.path.exists(SPLIT))

In [ ]:
# Integrity: every session in one partition, and every listed image present
import pandas as pd
from pathlib import Path
from src.train_teacher import check_split
counts = check_split(SPLIT)
table = pd.read_csv(SPLIT)
present = {p.relative_to(CROPS).with_suffix("").as_posix() for p in Path(CROPS).rglob("*.jpg")}
table["present"] = table["path"].str.replace(r"\.[^.]+$", "", regex=True).isin(present)
print("split counts:", counts)
print(table.groupby(["split", "class"])["present"].agg(["sum", "count"]))
assert table["present"].all(), "Some images listed in sessions.csv are missing from crops/"
print("sessions:", table["session"].nunique(), "| largest session:", table.groupby("session").size().max())

In [ ]:
# Look at a few training crops per class (sanity check of the inputs the teacher will see)
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for r, cls in enumerate(sorted(table["class"].unique())):
    sample = table[(table["split"] == "train") & (table["class"] == cls)].sample(8, random_state=0)
    for ax, rel in zip(axes[r], sample["path"]):
        ax.imshow(Image.open(Path(CROPS) / rel)); ax.axis("off")
    axes[r, 0].set_title(cls, loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 3. Smoke test
Three batches per epoch for one epoch — catches path, memory and download problems in a minute before the long runs.

In [ ]:
!python -m src.train_teacher train --arch efficientnet_b0 --data {CROPS} --split-file {SPLIT} \
    --out /content/smoke --epochs 1 --batch-size 16 --num-workers {WORKERS} --smoke --restart

## 4. Train the candidates
Each run writes `best.pt` (best validation macro-F1), `last.pt` (resume point), `history.csv` and `summary.json`
under `RUNS_DIR/<arch>/`. Re-run this cell after a disconnect to resume.

In [ ]:
for arch in ARCHS:
    print("=" * 30, arch, "=" * 30)
    !python -m src.train_teacher train --arch {arch} --data {CROPS} --split-file {SPLIT} --out {RUNS_DIR} \
        --epochs {EPOCHS} --batch-size {BATCH} --num-workers {WORKERS}

In [ ]:
# Learning curves (validation only)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for arch in ARCHS:
    h = Path(RUNS_DIR) / arch / "history.csv"
    if not h.exists():
        continue
    df = pd.read_csv(h)
    axes[0].plot(df["epoch"] + 1, df["train_loss"], label=arch)
    axes[1].plot(df["epoch"] + 1, df["val_macro_f1"], label=arch)
    axes[2].plot(df["epoch"] + 1, df["val_recall_spoiled"], label=arch)
for ax, t in zip(axes, ["training loss", "validation macro-F1", "validation spoiled recall"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[1].legend(); plt.tight_layout(); plt.show()

## 5. Select the teacher (validation) and evaluate it once on test

In [ ]:
!python -m src.train_teacher select --data {CROPS} --split-file {SPLIT} --out {RUNS_DIR} --num-workers {WORKERS}

In [ ]:
import json
sel = json.load(open(Path(RUNS_DIR) / "teacher_selection.json"))
print("Selected teacher:", sel["selected_teacher"], "->", sel["checkpoint"])
display(pd.DataFrame(sel["validation_ranking"]).set_index("arch").round(4))
t = sel["test"]
print(f"Test ({sel['n_test_images']} images, {sel['n_test_sessions']} sessions): accuracy {t['accuracy']:.4f} | "
      f"macro-F1 {t['macro_f1']:.4f} (95% CI {t['macro_f1_ci95_sessions'][0]:.3f}-{t['macro_f1_ci95_sessions'][1]:.3f}) | "
      f"spoiled recall {t['recall_positive']:.4f} | ROC-AUC {t['roc_auc']:.4f}")
print("Confusion matrix [rows = true Fresh, Spoiled]:", t["confusion"])

## 6. Next step
Use the selected checkpoint as the teacher in `02_distill_compress_export.ipynb`:
`KDTrainer(student, teacher=sel["checkpoint"], ...)` — `KDTrainer` rebuilds the teacher from the checkpoint and
aligns the student's Stage-3 features with the teacher's penultimate features.

Record for the paper (Section 3.4 / Table 4): selected architecture, its parameter count, best epoch, the
validation ranking table, the GPU name (in `summary.json` → `meta`), and the test metrics above.